# Hands-on 30 (Bab 12): Pipeline Lengkap Sensor -> MQTT -> Python -> Prediksi -> Storage -> Dashboard

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 12.

Pipeline ujung ke ujung Smart Room Monitoring: simulator mengirim telemetry ke MQTT,
subscriber Python melakukan preprocessing (lag + moving average dari buffer), model
klasifikasi memberi status (normal/warning/fault), hasil disimpan. Di buku storage
memakai InfluxDB + dashboard Grafana; di sini didemokan dengan storage CSV lokal
(InfluxDB opsional via `--influx`, butuh Docker).

## Tujuan

Membangun satu pipeline IoT-ML lengkap yang benar-benar bekerja dari ujung ke ujung. Hasil akhir:

- mensimulasikan sensor yang mengirim data,
- menerima data melalui MQTT,
- melakukan preprocessing ringan di Python,
- menjalankan model untuk inferensi otomatis,
- menyimpan data mentah dan hasil prediksi,
- dan menampilkan hasil pada dashboard.


In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

# --- Utilitas notebook: menjalankan skrip buku APA ADANYA (tanpa flag) ---
# Skrip buku memakai while True / loop_forever, jadi notebook menghentikannya
# dari luar setelah beberapa detik dan menyimpan keluaran parsialnya.
import subprocess, sys, time
from pathlib import Path

LOG_DIR = Path("outputs/logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def latar(script, nama_log, args=()):
    """Jalankan skrip buku di latar belakang (python -u), keluaran ke file log."""
    f = open(LOG_DIR / nama_log, "w", encoding="utf-8")
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=f, stderr=subprocess.STDOUT)
    return p, f

def henti(*pasangan):
    """Hentikan proses latar belakang dan tutup file lognya."""
    for p, f in pasangan:
        p.terminate()
        try:
            p.wait(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
        f.close()

def jalankan_buku(script, detik, args=()):
    """Jalankan skrip buku selama `detik` detik lalu hentikan; kembalikan keluaran."""
    p = subprocess.Popen([sys.executable, "-u", script, *args],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        out, _ = p.communicate(timeout=detik)
    except subprocess.TimeoutExpired:
        p.terminate()
        try:
            out, _ = p.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            p.kill()
            out, _ = p.communicate()
    return out

def log_teks(nama_log):
    return (LOG_DIR / nama_log).read_text(encoding="utf-8")

# --- Utilitas InfluxDB (nilai bawaan skrip buku) ---
from influxdb_client import InfluxDBClient
INFLUX_URL, INFLUX_TOKEN = "http://localhost:8086", "GANTI_DENGAN_TOKEN_ANDA"
INFLUX_ORG, INFLUX_BUCKET = "iotlab", "sensor_data"
_ic = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)
if not _ic.ping():
    raise RuntimeError("InfluxDB belum berjalan di localhost:8086 (org iotlab, bucket sensor_data, "
                       "token GANTI_DENGAN_TOKEN_ANDA). Skrip buku di notebook ini menulis ke InfluxDB.")

def influx_df(measurement, rentang="-30d"):
    import pandas as pd
    q = (f'from(bucket:"{INFLUX_BUCKET}") |> range(start: {rentang}, stop: 1d) '
         f'|> filter(fn:(r) => r._measurement == "{measurement}") '
         f'|> pivot(rowKey:["_time"], columnKey:["_field"], valueColumn:"_value")')
    res = _ic.query_api().query_data_frame(q)
    df = pd.concat(res) if isinstance(res, list) else res
    if len(df):
        df = df.drop(columns=[c for c in df.columns if c in ("result", "table", "_start", "_stop", "_measurement")])
        df = df.sort_values("_time").reset_index(drop=True)
    return df

def influx_hapus(measurement):
    _ic.delete_api().delete("1970-01-01T00:00:00Z", "2100-01-01T00:00:00Z",
                            f'_measurement="{measurement}"', bucket=INFLUX_BUCKET, org=INFLUX_ORG)


[INFO] Working directory: /downloads/iot-ml-book


## 1. Jalankan Pipeline End-to-End (broker + subscriber + simulator)

In [2]:
influx_hapus("room_pipeline")
broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = latar("scripts/pipeline_room_subscriber.py", "pipeline_sub_nb.log")
    time.sleep(2)
    pub = latar("scripts/publisher_room_simulator.py", "pipeline_pub_nb.log")
    time.sleep(24)   # sekitar 12 pesan (interval 2 detik)
    henti(pub, sub)
    print("--- Output publisher ---")
    print(log_teks("pipeline_pub_nb.log")[:600])
    print("--- Output subscriber pipeline ---")
    print(log_teks("pipeline_sub_nb.log"))
finally:
    broker.terminate()

--- Output publisher ---
Published: {"timestamp": "2026-11-02T12:17:26", "device_id": "room01", "temperature_c": 26.37, "humidity_pct": 50.04, "co2_ppm": 1433, "power_watt": 161.09}
Published: {"timestamp": "2026-11-02T12:17:28", "device_id": "room01", "temperature_c": 27.81, "humidity_pct": 64.21, "co2_ppm": 963, "power_watt": 173.85}
Published: {"timestamp": "2026-11-02T12:17:30", "device_id": "room01", "temperature_c": 27.07, "humidity_pct": 45.24, "co2_ppm": 1174, "power_watt": 141.05}
Published: {"timestamp": "2026-11-02T12:17:32", "device_id": "room01", "temperature_c": 29.39, "humidity_pct": 48.27, "co2_ppm": 9
--- Output subscriber pipeline ---
Subscriber terhubung ke broker.
Subscribe ke: iotlab/room01/telemetry
[2026-11-02T12:17:26] T=26.37 H=50.04 CO2=1433 P=161.09 -> status=normal
[2026-11-02T12:17:28] T=27.81 H=64.21 CO2=963 P=173.85 -> status=normal
[2026-11-02T12:17:30] T=27.07 H=45.24 CO2=1174 P=141.05 -> status=normal
[2026-11-02T12:17:32] T=29.39 H=48.27 CO2=904 P=227

## 2. Verifikasi Hasil di Storage CSV

In [3]:
df = influx_df("room_pipeline")
print(df[["_time", "temperature_c", "humidity_pct", "co2_ppm", "power_watt", "pred_status"]].to_string(index=False))
print("\nDistribusi status:", df["pred_status"].value_counts().to_dict())

                    _time  temperature_c  humidity_pct  co2_ppm  power_watt pred_status
2026-11-02 12:17:26+00:00          26.37         50.04   1433.0      161.09      normal
2026-11-02 12:17:28+00:00          27.81         64.21    963.0      173.85      normal
2026-11-02 12:17:30+00:00          27.07         45.24   1174.0      141.05      normal
2026-11-02 12:17:32+00:00          29.39         48.27    904.0      227.35     warning
2026-11-02 12:17:34+00:00          29.86         40.41    496.0      106.57     warning
2026-11-02 12:17:36+00:00          25.41         48.06   1237.0      255.02       fault
2026-11-02 12:17:38+00:00          24.19         42.87   1094.0      162.95      normal
2026-11-02 12:17:40+00:00          33.48         40.83    618.0      234.79     warning
2026-11-02 12:17:42+00:00          31.13         49.28   1175.0      215.77     warning
2026-11-02 12:17:44+00:00          30.49         40.34   1488.0      191.68     warning
2026-11-02 12:17:46+00:00       

## Penutup

Seluruh komponen menyatu: MQTT menjadi pintu masuk pipeline analitik, subscriber
Python menjadi pusat logika (validasi, fitur lag/rolling dari buffer), model dipanggil
otomatis tanpa notebook, dan hasil inferensi tersimpan. Untuk deployment penuh sesuai
buku, jalankan InfluxDB 2 dan Grafana via Docker, aktifkan `--influx`, lalu susun panel
suhu/kelembaban/CO2/daya/status dengan ambang warna normal-hijau, warning-kuning,
fault-merah.